# Customer Support Ticket Analytics System
## Data Cleaning

**Dataset:** `Raw_Data_set_.csv`  
**Rows:** 200,000 | **Columns:** 30



In [ ]:
import pandas as pd
import numpy as np


pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

---
## 1. Load & Initial Inspection

In [4]:
df = pd.read_csv('c:/Users/Tarun/OneDrive/Desktop/Project 3/Data Sets/Raw Data set .csv')
print(f'Shape: {df.shape}')
df.head(3)

Shape: (200000, 30)


,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,region,customer_age,customer_gender,subscription_type,customer_tenure_months,previous_tickets,customer_satisfaction_score,first_response_time_hours,resolution_time_hours,ticket_created_date,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,North America,31,Male,Free,36,6,5,30.50,108.36,2023-05-17,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,South America,23,Other,Premium,54,20,5,63.81,87.43,2024-01-06,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Chat,Europe,47,Female,Premium,60,20,5,16.20,78.50,2022-11-30,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 30 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ticket_id                    200000 non-null  int64  
 1   customer_name                200000 non-null  object 
 2   customer_email               200000 non-null  object 
 3   product                      200000 non-null  object 
 4   category                     200000 non-null  object 
 5   issue_description            200000 non-null  object 
 6   resolution_notes             200000 non-null  object 
 7   priority                     200000 non-null  object 
 8   status                       200000 non-null  object 
 9   channel                      200000 non-null  object 
 10  region                       200000 non-null  object 
 11  customer_age                 200000 non-null  int64  
 12  customer_gender              200000 non-null  object 
 13 

In [6]:
df.describe()

,ticket_id,customer_age,customer_tenure_months,previous_tickets,customer_satisfaction_score,first_response_time_hours,resolution_time_hours,issue_complexity_score
count,200000.00,200000.00,200000.00,200000.00,200000.00,200000.00,200000.00,200000.00
mean,100000.50,46.47,30.38,9.99,3.00,36.31,120.54,5.50
std,57735.17,16.75,17.33,6.05,1.41,20.65,68.97,2.88
min,1.00,18.00,1.00,0.00,1.00,0.50,1.00,1.00
25%,50000.75,32.00,15.00,5.00,2.00,18.46,60.81,3.00
50%,100000.50,46.00,30.00,10.00,3.00,36.31,120.45,6.00
75%,150000.25,61.00,45.00,15.00,4.00,54.22,180.26,8.00
max,200000.00,75.00,60.00,20.00,5.00,72.00,240.00,10.00


In [7]:

missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
missing[missing['missing_count'] > 0]

,missing_count,missing_pct
browser,40023,20.01


In [8]:
# Duplicate check
print('Fully duplicate rows :', df.duplicated().sum())
print('Duplicate ticket_ids :', df.duplicated(subset=['ticket_id']).sum())

Fully duplicate rows : 0
Duplicate ticket_ids : 0


---
## 2. Fix Data Types

- `ticket_created_date` and `ticket_resolved_date` → `datetime64`
- `escalated` and `sla_breached` (Yes/No strings) → `bool`
- `customer_satisfaction_score` and `issue_complexity_score` → stay `int` (ordinal, finite range)

In [9]:
# Parse dates
df['ticket_created_date']  = pd.to_datetime(df['ticket_created_date'])
df['ticket_resolved_date'] = pd.to_datetime(df['ticket_resolved_date'])

# Convert Yes/No columns to boolean
bool_map = {'Yes': True, 'No': False}
df['escalated']    = df['escalated'].map(bool_map)
df['sla_breached'] = df['sla_breached'].map(bool_map)

print('Date dtypes:')
print(df[['ticket_created_date', 'ticket_resolved_date']].dtypes)
print('\nBoolean dtypes:')
print(df[['escalated', 'sla_breached']].dtypes)

Date dtypes:
ticket_created_date     datetime64[ns]
ticket_resolved_date    datetime64[ns]
dtype: object

Boolean dtypes:
escalated       bool
sla_breached    bool
dtype: object


---
## 3. Handle Missing Values

`browser` is the **only column with nulls** (~20% missing, 40,023 rows).  
Missing browser likely means the customer did not use a web browser (e.g., Phone / Mobile App channel).  
We fill with `'Unknown'` — dropping 20% of the data would be too aggressive.

In [10]:
# Cross-tab: is browser null correlated with channel?
df['browser_missing'] = df['browser'].isnull()
print(df.groupby('channel')['browser_missing'].mean().sort_values(ascending=False).to_string())
df.drop(columns=['browser_missing'], inplace=True)

channel
Social Media   0.20
Phone          0.20
Chat           0.20
Web Form       0.20
Email          0.20


In [11]:
# Fill missing browser with 'Unknown'
df['browser'] = df['browser'].fillna('Unknown')

print('Remaining nulls in browser:', df['browser'].isnull().sum())

Remaining nulls in browser: 0


---
## 4. Standardise Categorical Columns



In [12]:
str_cols = df.select_dtypes(include='object').columns.tolist()
print('String columns:', str_cols)

String columns: ['customer_name', 'customer_email', 'product', 'category', 'issue_description', 'resolution_notes', 'priority', 'status', 'channel', 'region', 'customer_gender', 'subscription_type', 'operating_system', 'browser', 'payment_method', 'language', 'preferred_contact_time', 'customer_segment']


In [13]:
for col in str_cols:
    df[col] = df[col].str.strip()

# Title-case the columns where casing matters for readability
title_cols = ['priority', 'status', 'channel', 'region', 'customer_gender',
              'subscription_type', 'operating_system', 'browser',
              'payment_method', 'language', 'preferred_contact_time', 'customer_segment']

for col in title_cols:
    df[col] = df[col].str.title()

# Verify unique values look clean
for col in title_cols:
    print(f'{col:30s}: {sorted(df[col].unique().tolist())}')

priority                      : ['High', 'Low', 'Medium', 'Urgent']
status                        : ['Closed', 'In Progress', 'Open', 'Pending Customer', 'Resolved']
channel                       : ['Chat', 'Email', 'Phone', 'Social Media', 'Web Form']
region                        : ['Africa', 'Asia', 'Australia', 'Europe', 'North America', 'South America']
customer_gender               : ['Female', 'Male', 'Other']
subscription_type             : ['Basic', 'Enterprise', 'Free', 'Premium']
operating_system              : ['Android', 'Ios', 'Linux', 'Macos', 'Windows']
browser                       : ['Chrome', 'Edge', 'Firefox', 'Safari', 'Unknown']
payment_method                : ['Bank Transfer', 'Credit Card', 'Crypto', 'Debit Card', 'Paypal']
language                      : ['Chinese', 'English', 'French', 'German', 'Japanese', 'Spanish']
preferred_contact_time        : ['Afternoon', 'Evening', 'Morning', 'Night']
customer_segment              : ['Corporate', 'Individual', 'Small 

---
## 5. Fix Logical Inconsistency — Open Tickets with a Resolved Date


In [14]:
open_with_date_mask = (df['status'] == 'Open') & (df['ticket_resolved_date'].notna())
print('Open tickets with a resolved date (before fix):', open_with_date_mask.sum())

df.loc[open_with_date_mask, 'ticket_resolved_date'] = pd.NaT

print('Open tickets with a resolved date (after fix) :', 
      ((df['status'] == 'Open') & (df['ticket_resolved_date'].notna())).sum())

Open tickets with a resolved date (before fix): 39922
Open tickets with a resolved date (after fix) : 0


---
## 6. Feature Engineering



In [16]:

df['actual_resolution_days'] = (
    (df['ticket_resolved_date'] - df['ticket_created_date'])
    .dt.total_seconds() / 86400
).round(2)

df['created_year']  = df['ticket_created_date'].dt.year
df['created_month'] = df['ticket_created_date'].dt.month
df['created_day_of_week'] = df['ticket_created_date'].dt.day_name()

def response_bucket(hours):
    if hours <= 1:
        return 'Immediate (<1h)'
    elif hours <= 8:
        return 'Same Day (<8h)'
    elif hours <= 24:
        return 'Next Day (<24h)'
    else:
        return 'Delayed (>24h)'

df['response_speed_bucket'] = df['first_response_time_hours'].apply(response_bucket)

print('New columns added:')
print(df[['actual_resolution_days', 'created_year', 'created_month',
          'created_day_of_week', 'response_speed_bucket']].head(5))

New columns added:
   actual_resolution_days  created_year  created_month created_day_of_week  \
0                     NaN          2023              5           Wednesday   
1                   13.00          2024              1            Saturday   
2                    5.00          2022             11           Wednesday   
3                   14.00          2024              3            Thursday   
4                    8.00          2024              8              Friday   

  response_speed_bucket  
0        Delayed (>24h)  
1        Delayed (>24h)  
2       Next Day (<24h)  
3        Delayed (>24h)  
4        Delayed (>24h)  


---
## 7. Final Validation

In [17]:
print('=== Final Shape ===')
print(df.shape)

print('\n=== Remaining Nulls ===')
null_summary = df.isnull().sum()
print(null_summary[null_summary > 0])
if null_summary.sum() == 0:
    print('No unexpected nulls remaining (ticket_resolved_date NaT for open tickets is expected).')

print('\n=== Numeric Range Check ===')
print(df[['customer_age', 'customer_satisfaction_score', 'issue_complexity_score',
          'first_response_time_hours', 'resolution_time_hours']].describe().round(2))

print('\n=== Data Types ===')
print(df.dtypes)

=== Final Shape ===
(200000, 35)

=== Remaining Nulls ===
ticket_resolved_date      39922
actual_resolution_days    39922
dtype: int64

=== Numeric Range Check ===
       customer_age  customer_satisfaction_score  issue_complexity_score  \
count     200000.00                    200000.00               200000.00   
mean          46.47                         3.00                    5.50   
std           16.75                         1.41                    2.88   
min           18.00                         1.00                    1.00   
25%           32.00                         2.00                    3.00   
50%           46.00                         3.00                    6.00   
75%           61.00                         4.00                    8.00   
max           75.00                         5.00                   10.00   

       first_response_time_hours  resolution_time_hours  
count                  200000.00              200000.00  
mean                       36.31   

In [18]:
# Final preview
df.head(3)

,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,region,customer_age,customer_gender,subscription_type,customer_tenure_months,previous_tickets,customer_satisfaction_score,first_response_time_hours,resolution_time_hours,ticket_created_date,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment,actual_resolution_days,created_year,created_month,created_day_of_week,response_speed_bucket
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,North America,31,Male,Free,36,6,5,30.50,108.36,2023-05-17,NaT,False,True,Macos,Edge,Paypal,French,Afternoon,4,Small Business,NaN,2023,5,Wednesday,Delayed (>24h)
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,South America,23,Other,Premium,54,20,5,63.81,87.43,2024-01-06,2024-01-19,True,True,Windows,Firefox,Paypal,English,Afternoon,2,Small Business,13.00,2024,1,Saturday,Delayed (>24h)
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Chat,Europe,47,Female,Premium,60,20,5,16.20,78.50,2022-11-30,2022-12-05,True,True,Windows,Safari,Bank Transfer,French,Morning,4,Corporate,5.00,2022,11,Wednesday,Next Day (<24h)


In [23]:
output_path = 'C:/Users/Tarun/OneDrive/Desktop/Project 3/Data Sets/Clean_Dataset.csv'
df.to_csv(output_path, index=False)
print(f'Cleaned dataset saved → {output_path}')
print(f'Final shape: {df.shape}')

Cleaned dataset saved → C:/Users/Tarun/OneDrive/Desktop/Project 3/Data Sets/Clean_Dataset.csv
Final shape: (200000, 35)
